# NB-08 — T5 paper trading engine walkthrough

Ships alongside T5 phase P3.a (#1755) of #1719.

## What this notebook covers

- Why the paper engine exists (Fidelity has no API, so the operator files
  orders manually; this ledger shadows the real book)
- Opening `SqlitePaperEngine` and understanding its data model
- Submitting a batch and inspecting **PENDING** orders
- Recording fills — **PENDING → PARTIAL → FILLED** transitions
- **FIFO realized P&L** on a two-lot close
- **Cancel** semantics + idempotency
- **Loud rejection matrix** — every impossible state raises `PaperEngineError`
- **Persistence** across process restarts

## What this notebook does NOT cover

- The CSV / XLSX write path — see **NB-07** (already merged)
- The full end-to-end flow — see **NB-09**
- Unrealized P&L (needs live prices — P3.b)
- Reconciliation vs. Fidelity's real position snapshot (P3.c)

Full test guide at `docs/superpowers/specs/2026-08-03-t5-e2e-test-guide.md`.


## 1. Motivation

Fidelity has no API. Every order the operator wants filled must be
typed manually into Fidelity's order-entry UI. The paper trading
engine's job is to **shadow that real book** — keeping track of what
was ordered, what was actually filled, and the resulting cash /
position / realized-P&L consequences.

Two rules govern the engine:

1. **Nothing auto-fills.** The operator (in future: a P5 Activity CSV
   parser) records each fill after Fidelity confirms it.
2. **Every impossible state raises loudly.** A fill on an unknown
   order, an overfill, a sell that would exceed the long position, or
   a buy that would drive cash negative — all raise
   `PaperEngineError`. No silent coercion, no partial writes.

That discipline turns the paper engine into a *sanity check* against
operator mistakes at Fidelity's UI. If you record a fill of 200 shares
against a 100-share order, the engine catches it before your local
book drifts from the real one.


## 2. Opening the engine

`SqlitePaperEngine` stores state in a caller-supplied SQLite file. For
this notebook we use a per-run temp file. In real usage the operator
points at `~/.portfolio_intel/paper.db` (the factory default) so state
survives across sessions.


In [ ]:
import tempfile
from decimal import Decimal
from pathlib import Path
from openbb_techtrade.execution.paper_engine import SqlitePaperEngine

# Per-run temp DB so re-executing the notebook always starts clean.
db_path = Path(tempfile.mkdtemp(prefix="nb08_")) / "paper.db"
engine = SqlitePaperEngine(db_path, starting_cash=Decimal("100000"))

acct = engine.get_account()
print(f"Account:        {acct.account_id}")
print(f"Starting cash:  {acct.starting_cash}")
print(f"Cash:           {acct.cash}")
print(f"Realized P&L:   {acct.realized_pl}")

Account:        paper
Starting cash:  100000
Cash:           100000
Realized P&L:   0


## 3. Submit a batch → PENDING orders

The engine accepts an `OrderBatch` (from NB-07) and records each
ticket as a PENDING order. `submit_batch` returns the assigned
`order_id` list — the operator uses these when recording fills.


In [ ]:
from openbb_techtrade.execution.order_sink import OrderBatch, OrderTicket

batch = OrderBatch(tickets=(
    OrderTicket(symbol="MSFT",  action="Buy",  quantity=Decimal("50"),
                order_type="Limit", limit_price=Decimal("400.00")),
    OrderTicket(symbol="AAPL",  action="Buy",  quantity=Decimal("100"),
                order_type="Limit", limit_price=Decimal("180.00")),
    OrderTicket(symbol="GOOGL", action="Buy",  quantity=Decimal("20"),
                order_type="Market"),
))
order_ids = engine.submit_batch(batch, plan_id="nb08-demo")
print(f"{len(order_ids)} orders submitted:")
for oid in order_ids:
    print(f"  {oid}")

3 orders submitted:
  ord_790e9dd61c64
  ord_5e4018260ac0
  ord_0ef6be545164


In [ ]:
from openbb_techtrade.execution.paper_engine import OrderStatus

pending = engine.get_orders(status=OrderStatus.PENDING)
print(f"{len(pending)} PENDING order(s):")
for o in pending:
    print(f"  {o.symbol:6} {o.side.value:5} qty={o.quantity} type={o.order_type} limit={o.limit_price}")

3 PENDING order(s):
  MSFT   BUY   qty=50 type=Limit limit=400.00
  AAPL   BUY   qty=100 type=Limit limit=180.00
  GOOGL  BUY   qty=20 type=Market limit=None


## 4. Record a full fill on order 1 → FILLED

The operator files the MSFT order at Fidelity, gets a confirmation
saying "filled 50 shares at $398.50 with $1 commission", and calls
`record_fill()`. The engine transitions the order to FILLED, debits
cash, and creates a position.


In [ ]:
from datetime import datetime, timezone

# MSFT — full fill at 398.50 with $1 commission.
fill = engine.record_fill(
    order_ids[0],
    price=Decimal("398.50"),
    filled_qty=Decimal("50"),
    at=datetime.now(timezone.utc),
    commission=Decimal("1"),
)
print(f"Fill recorded:")
print(f"  fill_id:      {fill.fill_id}")
print(f"  symbol/side:  {fill.symbol}/{fill.side.value}")
print(f"  filled_qty:   {fill.filled_qty}")
print(f"  price:        {fill.price}")
print(f"  commission:   {fill.commission}")

# Order status flipped to FILLED.
msft_order = [o for o in engine.get_orders() if o.symbol == "MSFT"][0]
print(f"\nMSFT order status: {msft_order.status.value}")

# Cash decreased by 50 * 398.50 + 1 = 19926.
print(f"Cash: {engine.get_account().cash}  (100000 - 19926 = 80074)")

Fill recorded:
  fill_id:      fil_c70f79762245
  symbol/side:  MSFT/BUY
  filled_qty:   50
  price:        398.50
  commission:   1

MSFT order status: FILLED
Cash: 80074.00  (100000 - 19926 = 80074)


## 5. Partial fill on order 2 → PARTIAL, then FILLED

The AAPL order comes back in two pieces from Fidelity: 60 shares at
$179.75 first, then 40 shares at $180.10 (probably crossed price
levels during execution). The engine handles partial fills
naturally — first fill flips status to PARTIAL, second fill completes it.


In [ ]:
# First partial fill (60 of 100).
engine.record_fill(order_ids[1], price=Decimal("179.75"),
                   filled_qty=Decimal("60"), at=datetime.now(timezone.utc),
                   commission=Decimal("1"))
aapl_order = [o for o in engine.get_orders() if o.symbol == "AAPL"][0]
print(f"After 1st fill: AAPL status = {aapl_order.status.value}")

# Second fill completes the order.
engine.record_fill(order_ids[1], price=Decimal("180.10"),
                   filled_qty=Decimal("40"), at=datetime.now(timezone.utc),
                   commission=Decimal("1"))
aapl_order = [o for o in engine.get_orders() if o.symbol == "AAPL"][0]
print(f"After 2nd fill: AAPL status = {aapl_order.status.value}")

# Weighted-average cost basis: (60*179.75 + 40*180.10) / 100 = 179.89
[aapl_pos] = [p for p in engine.get_positions() if p.symbol == "AAPL"]
print(f"\nAAPL position quantity: {aapl_pos.quantity}")
print(f"AAPL avg_cost:          {aapl_pos.avg_cost}  (weighted average)")

After 1st fill: AAPL status = PARTIAL
After 2nd fill: AAPL status = FILLED

AAPL position quantity: 100
AAPL avg_cost:          179.89  (weighted average)


## 6. Cancel order 3 → CANCELLED, then idempotent

The operator decides the GOOGL order isn't worth placing after all
and cancels it in the engine. Cancels on PENDING orders always
succeed; a second cancel on the same order logs but doesn't raise
(idempotent by contract).


In [ ]:
engine.cancel_order(order_ids[2], reason="Reviewed limit again, dropping")
googl_order = [o for o in engine.get_orders() if o.symbol == "GOOGL"][0]
print(f"GOOGL status: {googl_order.status.value}")

# Second cancel — idempotent, no raise.
engine.cancel_order(order_ids[2], reason="Retry from operator")
print("Second cancel: no exception (idempotent)")

# But cancel on a FILLED order DOES raise — that's a real error.
from openbb_techtrade.execution.paper_engine import PaperEngineError
try:
    engine.cancel_order(order_ids[0], reason="Too late")
except PaperEngineError as e:
    print(f"Cancel on FILLED order raised: {e}")

GOOGL status: CANCELLED
Second cancel: no exception (idempotent)
Cancel on FILLED order raised: cancel_order: order 'ord_790e9dd61c64' is FILLED; cannot cancel a terminated order


## 7. FIFO realized P&L — a two-lot close

Now the interesting math. The operator adds more MSFT at a different
price, then partially closes the position — the paper engine walks
open lots in **FIFO** (first-in-first-out) order and computes realized
P&L accurately.


In [ ]:
# Set up two lots: existing 50 @ 398.50, plus a fresh 20 @ 420.
[b2] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="MSFT", action="Buy", quantity=Decimal("20"),
                order_type="Market"),
)))
engine.record_fill(b2, price=Decimal("420.00"),
                   filled_qty=Decimal("20"), at=datetime.now(timezone.utc))

# Now sell 55 @ 430 — this will close the entire first lot (50 shares)
# plus 5 shares of the second lot.
[s1] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="MSFT", action="Sell", quantity=Decimal("55"),
                order_type="Market"),
)))
engine.record_fill(s1, price=Decimal("430.00"),
                   filled_qty=Decimal("55"), at=datetime.now(timezone.utc))

# FIFO math:
#   50 shares * (430 - 398.50) = 50 * 31.50 = 1575
#    5 shares * (430 - 420.00) =  5 * 10.00 =   50
#                                              ----
#                                             1625
[msft_pos] = [p for p in engine.get_positions() if p.symbol == "MSFT"]
print(f"MSFT position after 55-share sell: {msft_pos.quantity} shares")
print(f"MSFT avg_cost (FIFO, unchanged on close): {msft_pos.avg_cost}")

# Actually: after closing all 50 of lot 1, only lot 2 (originally 20, now 15)
# remains — so avg_cost resets to 420.
# Verify math: sell 55 out of 70 → 15 remaining; realized 1625.
print(f"MSFT realized P&L: {msft_pos.realized_pl}")
print(f"\nAccount realized P&L: {engine.get_account().realized_pl}")

MSFT position after 55-share sell: 15 shares
MSFT avg_cost (FIFO, unchanged on close): 404.6428571428571428571428571
MSFT realized P&L: 1625.00

Account realized P&L: 1625.00


## 8. Loud rejection matrix

Every impossible state raises `PaperEngineError`. Below we demonstrate
five of them. Each raise is the engine catching a real-world mistake
before it corrupts the local book.


In [ ]:
# D1 — unknown order_id
try:
    engine.record_fill("ord_does_not_exist",
                       price=Decimal("100"), filled_qty=Decimal("1"),
                       at=datetime.now(timezone.utc))
except PaperEngineError as e:
    print(f"D1 unknown order_id:")
    print(f"    {e}\n")

D1 unknown order_id:
    record_fill: unknown order_id 'ord_does_not_exist'



In [ ]:
# D2 — overfill: fill exceeds ordered quantity
[oid] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="NVDA", action="Buy", quantity=Decimal("10"), order_type="Market"),
)))
try:
    engine.record_fill(oid, price=Decimal("130"),
                       filled_qty=Decimal("11"),  # 11 > 10 ordered
                       at=datetime.now(timezone.utc))
except PaperEngineError as e:
    print(f"D2 overfill:")
    print(f"    {e}\n")

# Cleanup: cancel the NVDA order so it doesn't linger.
engine.cancel_order(oid, reason="D2 demo cleanup")

D2 overfill:
    record_fill: overfill on 'ord_c18b70dfaa1f' — ordered=10, prior=0, this=11 → would total 11



In [ ]:
# D3 — sell exceeds long position (no accidental short)
# Try to sell more MSFT than we hold.
current_msft = [p for p in engine.get_positions() if p.symbol == "MSFT"][0].quantity
[sell_id] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="MSFT", action="Sell",
                quantity=current_msft + Decimal("100"),  # more than we have
                order_type="Market"),
)))
try:
    engine.record_fill(sell_id, price=Decimal("430"),
                       filled_qty=current_msft + Decimal("100"),
                       at=datetime.now(timezone.utc))
except PaperEngineError as e:
    print(f"D3 sell exceeds long position:")
    print(f"    {e}\n")

engine.cancel_order(sell_id, reason="D3 demo cleanup")

D3 sell exceeds long position:
    _apply_fifo: sell of 115 MSFT exceeds long position by 100. Short-sell? Open a SELL_SHORT order first.



In [ ]:
# D4 — buy exceeds available cash
current_cash = engine.get_account().cash
[huge_buy] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="NVDA", action="Buy", quantity=Decimal("10000"),
                order_type="Market"),
)))
try:
    # 10_000 * 130 = 1_300_000 — WAY over cash.
    engine.record_fill(huge_buy, price=Decimal("130"),
                       filled_qty=Decimal("10000"),
                       at=datetime.now(timezone.utc))
except PaperEngineError as e:
    print(f"D4 cash would go negative:")
    print(f"    {e}\n")

engine.cancel_order(huge_buy, reason="D4 demo cleanup")

D4 cash would go negative:
    cash side effect: would drive cash negative (77333.00 + -1300000 = -1222667.00). Buy exceeds available cash — check the batch's gross notional against get_account().cash before submitting.



In [ ]:
# D5 — fill on cancelled order
[cx_id] = engine.submit_batch(OrderBatch(tickets=(
    OrderTicket(symbol="AAPL", action="Buy", quantity=Decimal("1"), order_type="Market"),
)))
engine.cancel_order(cx_id, reason="D5 setup")
try:
    engine.record_fill(cx_id, price=Decimal("180"),
                       filled_qty=Decimal("1"), at=datetime.now(timezone.utc))
except PaperEngineError as e:
    print(f"D5 fill on CANCELLED order:")
    print(f"    {e}")

D5 fill on CANCELLED order:
    record_fill: order 'ord_a8c176a81230' is CANCELLED; cannot record fill on a terminated order


Each of these keeps the local book aligned with the real one at
Fidelity. If Fidelity confirms an overfill, that's a broker bug — you
want to catch it here before your position math drifts.


## 9. Persistence across process restart

The engine's SQLite DB survives process restarts by design. Same
`db_path` = same account, same cash, same positions, same fill log.
The `starting_cash` argument is honored only on the FIRST open
(account creation); subsequent opens ignore it.


In [ ]:
# Capture current state.
cash_before = engine.get_account().cash
positions_before = {p.symbol: p.quantity for p in engine.get_positions()}
fills_before = len(engine.get_fills())

# Close the engine (releases the SQLite connection).
engine.close()

# Re-open with an intentionally-different starting_cash — must be IGNORED.
engine2 = SqlitePaperEngine(db_path, starting_cash=Decimal("999999"))
acct2 = engine2.get_account()
print(f"After restart:")
print(f"  Cash                = {acct2.cash}  (expected: {cash_before})")
print(f"  starting_cash arg ignored — account.starting_cash = {acct2.starting_cash}")
print(f"  Positions preserved = {dict((p.symbol, p.quantity) for p in engine2.get_positions())}")
print(f"  Fill count preserved = {len(engine2.get_fills())}  (expected: {fills_before})")
engine2.close()

After restart:
  Cash                = 77333.00  (expected: 77333.00)
  starting_cash arg ignored — account.starting_cash = 100000
  Positions preserved = {'AAPL': Decimal('100'), 'MSFT': Decimal('15')}
  Fill count preserved = 5  (expected: 5)


## 10. What's next

- **NB-07** (`07-t5-order-batch-and-xlsx-workbook.ipynb`) — the write
  side of T5: OrderTicket, OrderBatch, and the 6-sheet XLSX artifact
- **NB-09** (`09-t5-end-to-end-plan-to-fills.ipynb`) — the full flow
  from validated plan through XLSX artifact through recorded fills

For the full E2E test guide (QA + automation scenarios), see
`docs/superpowers/specs/2026-08-03-t5-e2e-test-guide.md`.
